# M0 - Baseline regression lineaire (5 variables)

Objectif:
- entrainer une baseline lineaire sur `train.csv`
- utiliser uniquement les 5 variables retenues dans l'analyse
- appliquer un preprocessing propre via pipeline sklearn
- generer un fichier de soumission Kaggle a partir de `test.csv`

Variables retenues:
`['OverallQual', 'GrLivArea', 'TotalBsmtSF', 'AgeAtSale', 'Neighborhood']`


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)


In [2]:
# Chargement des donnees
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "Data"

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"train shape: {train_df.shape}")
print(f"test shape: {test_df.shape}")
print(f"sample_submission shape: {sample_submission.shape}")


train shape: (1460, 81)
test shape: (1459, 80)
sample_submission shape: (1459, 2)


In [3]:
# Feature engineering: AgeAtSale = annee de vente - annee de construction
for df in [train_df, test_df]:
    df["AgeAtSale"] = df["YrSold"] - df["YearBuilt"]

features = ["OverallQual", "GrLivArea", "TotalBsmtSF", "AgeAtSale", "Neighborhood"]
target = "SalePrice"

print("Features baseline:", features)


Features baseline: ['OverallQual', 'GrLivArea', 'TotalBsmtSF', 'AgeAtSale', 'Neighborhood']


In [4]:
# Verification des valeurs manquantes pour les 5 variables
missing_train = train_df[features].isna().mean().mul(100).sort_values(ascending=False)
missing_test = test_df[features].isna().mean().mul(100).sort_values(ascending=False)

print("Missing % (train):")
print(missing_train)
print("\nMissing % (test):")
print(missing_test)


Missing % (train):
OverallQual     0.0
GrLivArea       0.0
TotalBsmtSF     0.0
AgeAtSale       0.0
Neighborhood    0.0
dtype: float64

Missing % (test):
TotalBsmtSF     0.06854
OverallQual     0.00000
GrLivArea       0.00000
AgeAtSale       0.00000
Neighborhood    0.00000
dtype: float64


In [5]:
# X / y
X_train = train_df[features].copy()
X_test = test_df[features].copy()

y_train = train_df[target].copy()
y_train_log = np.log1p(y_train)


In [6]:
# Preprocessing avec regles metier sur les manquants:
# - TotalBsmtSF manquant => 0 (interprete comme pas de basement)
# - Neighborhood manquant => categorie explicite "Missing"
# - autres numeriques => mediane
num_median_features = ["OverallQual", "GrLivArea", "AgeAtSale"]
num_zero_features = ["TotalBsmtSF"]
cat_features = ["Neighborhood"]

numeric_median_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

numeric_zero_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num_median", numeric_median_transformer, num_median_features),
        ("num_zero", numeric_zero_transformer, num_zero_features),
        ("cat", categorical_transformer, cat_features),
    ]
)

model = LinearRegression()

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)


In [7]:
# Evaluation en CV sur log(SalePrice)
# Kaggle evalue le RMSE sur le log du prix, on s'aligne dessus.

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    pipeline,
    X_train,
    y_train_log,
    cv=cv,
    scoring=rmse_scorer,
)

cv_rmse = -cv_scores
print("CV RMSE (log1p) par fold:", np.round(cv_rmse, 5))
print(f"CV RMSE (log1p) mean: {cv_rmse.mean():.5f}")
print(f"CV RMSE (log1p) std : {cv_rmse.std():.5f}")


CV RMSE (log1p) par fold: [0.16109 0.14806 0.22744 0.16114 0.15779]
CV RMSE (log1p) mean: 0.17110
CV RMSE (log1p) std : 0.02857


In [8]:
# Entrainement final sur tout le train
pipeline.fit(X_train, y_train_log)

# Prediction sur test puis retour a l'echelle du prix
test_pred_log = pipeline.predict(X_test)
test_pred = np.expm1(test_pred_log)

# Securite numerique: eviter des valeurs negatives dues a l'approximation lineaire
test_pred = np.clip(test_pred, a_min=0, a_max=None)

print("Predictions test - min / max:", float(test_pred.min()), float(test_pred.max()))


Predictions test - min / max: 56977.57694797689 876599.4256773081


In [9]:
# Creation du fichier de soumission Kaggle
submission = sample_submission.copy()
submission["SalePrice"] = test_pred

output_path = BASE_DIR / "submission_m0_linear.csv"
submission.to_csv(output_path, index=False)

print(f"Submission ecrite: {output_path}")
submission.head()


Submission ecrite: submission_m0_linear.csv


,Id,SalePrice
0,1461,122042.371149
1,1462,156075.905608
2,1463,162423.989606
3,1464,178418.282000
4,1465,237542.646769
